In [15]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

SYSTEM_PROMPT = """
You are an NLP Internship Tutor.
- Explain clearly and step by step.
- Use friendly and concise tone.
- Focus on NLP and LangChain examples.
- If unsure, say you need more details.
- Always summarize the answer in 3-5 points.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

chain = prompt | llm

response = chain.invoke({"question": "What is RAG in NLP?"})
print(response.content)


RAG stands for Retrieval-Augmented Generation, a technique in Natural Language Processing (NLP) that combines retrieval-based methods with generative models. Here’s a step-by-step breakdown:

1. **Retrieval Component**: RAG first retrieves relevant documents or pieces of information from a large dataset or knowledge base. This is typically done using a search algorithm that finds the most relevant texts based on a given query.

2. **Generative Component**: After retrieving the relevant information, RAG uses a generative model (like GPT or BART) to create a coherent response or text. This model takes both the original query and the retrieved documents into account to generate a more informed and contextually relevant answer.

3. **Combining Strengths**: The key advantage of RAG is that it leverages the strengths of both retrieval and generation. The retrieval part ensures that the model has access to up-to-date and specific information, while the generative part allows for more natural 

In [16]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnablePassthrough, RunnableMap

# 1️⃣ الموديل
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# 2️⃣ البرومبت
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an NLP Internship Tutor. Be helpful, concise, and remember context."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 3️⃣ إنشاء الذاكرة
memory = ConversationBufferMemory(memory_key="history", return_messages=True)

# 4️⃣ دالة تشغّل الموديل وتحفظ التاريخ
def run_chat(question):
    # تحميل التاريخ الحالي من الميموري
    history = memory.load_memory_variables({})["history"]

    # نستخدم RunnableMap لتجميع المتغيرات اللي هنمررها للـ prompt
    chain = (
        RunnableMap({
            "history": lambda x: history,
            "input": RunnablePassthrough()
        })
        | prompt
        | llm
    )

    # استدعاء النموذج
    response = chain.invoke(question)
    print(f"AI: {response.content}\n")

    # حفظ الرسائل في الميموري
    memory.save_context({"input": question}, {"output": response.content})

# 5️⃣ جرّب التفاعل
run_chat("Hello, who are you?")
run_chat("Can you remind me what I just asked?")
run_chat("Now explain what is RAG in NLP")


AI: Hello! I'm your NLP Internship Tutor, here to help you with any questions or topics related to natural language processing. How can I assist you today?

AI: You asked, "Hello, who are you?" Would you like to ask anything else or discuss a specific topic?

AI: RAG stands for Retrieval-Augmented Generation. It is a model architecture that combines retrieval-based and generation-based approaches in natural language processing. 

In RAG, the model first retrieves relevant documents or pieces of information from a large corpus based on a given query. Then, it uses this retrieved information to generate a more informed and contextually relevant response. This approach leverages the strengths of both retrieval (for factual accuracy and grounding) and generation (for fluency and coherence).

RAG is particularly useful in tasks like question answering and conversational agents, where having access to external knowledge can significantly enhance the quality of the generated responses. Would 

In [17]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from pathlib import Path


In [18]:
# تأكد إن عندك مجلد للنصوص
data_dir = Path("data_docs")
data_dir.mkdir(exist_ok=True)

# مثال: ملف بسيط فيه معلومات
with open(data_dir / "rag_info.txt", "w", encoding="utf-8") as f:
    f.write("""
RAG (Retrieval Augmented Generation) is a method that combines retrieval and generation.
It retrieves relevant data from a database and uses it to improve the accuracy of language models.
Common evaluation metrics include Precision@K, Recall@K, MRR, and nDCG.
""")

# تحميل الملفات
loader = TextLoader(str(data_dir / "rag_info.txt"), encoding="utf-8")
docs = loader.load()
print("Documents loaded:", len(docs))


Documents loaded: 1


In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print("Chunks:", len(chunks))


Chunks: 1


In [20]:
# إنشاء نموذج تحويل النصوص لـ embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# تحديد مجلد التخزين المحلي
persist_directory = "chroma_store"

# إنشاء قاعدة بيانات Chroma
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

vector_db.persist()  # حفظ دائم على الجهاز
print("✅ Embeddings saved locally in:", persist_directory)


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Embeddings saved locally in: chroma_store


In [21]:
# إنشاء retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

query = "What are common RAG evaluation metrics?"
results = retriever.get_relevant_documents(query)

print("🔍 Retrieved documents:")
for i, r in enumerate(results, 1):
    print(f"{i}.", r.page_content)


🔍 Retrieved documents:
1. RAG (Retrieval Augmented Generation) is a method that combines retrieval and generation.
It retrieves relevant data from a database and uses it to improve the accuracy of language models.
Common evaluation metrics include Precision@K, Recall@K, MRR, and nDCG.
2. RAG (Retrieval Augmented Generation) is a method that combines retrieval and generation.
It retrieves relevant data from a database and uses it to improve the accuracy of language models.
Common evaluation metrics include Precision@K, Recall@K, MRR, and nDCG.


In [22]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question concisely:
{context}
Question: {question}
Answer in 3-5 lines.
""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

response = rag_chain.invoke("Explain RAG in simple words.")
print("🧠 RAG Response:", response.content)


🧠 RAG Response: RAG, or Retrieval Augmented Generation, is a technique that helps language models become more accurate by first finding relevant information from a database and then using that information to generate better responses. It combines two processes: retrieving data and generating text, making the output more informed and precise.


In [23]:
# task3_prep_docs.py
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

data_dir = Path("data_docs")
data_dir.mkdir(exist_ok=True)

# أمثلة ملفات (ضيف/عدّل على مزاجك)
(data_dir / "rag_intro.txt").write_text(
"""RAG combines retrieval with generation to ground LLM answers.
Common retrieval metrics: Precision@K, Recall@K, MRR, nDCG.
ai_definition.txt
Artificial intelligence (AI) is the simulation of human intelligence processes by machines.

""", encoding="utf-8"
)

(data_dir / "chroma_persist.txt").write_text(
"""Chroma provides persistent local vector storage using persist_directory.
It enables fast retrieval across sessions.
""", encoding="utf-8"
)

# تحميل + تقسيم مع doc_id
all_docs = []
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
for p in data_dir.glob("*.txt"):
    raw = TextLoader(str(p), encoding="utf-8").load()
    chunks = splitter.split_documents(raw)
    for i, d in enumerate(chunks):
        d.metadata["doc_id"] = p.stem  # مهم!
        d.metadata["chunk_id"] = i
    all_docs.extend(chunks)

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma.from_documents(
    all_docs, emb, persist_directory="chroma_store"
)
vector_db.persist()
print("✅ Ingested with doc_id metadata -> chroma_store/")


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Ingested with doc_id metadata -> chroma_store/


In [24]:
# task3_eval.py
from typing import List, Dict
from math import log2
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

DB_DIR = "chroma_store"
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = Chroma(persist_directory=DB_DIR, embedding_function=emb)
retriever = vs.as_retriever(search_kwargs={"k": 5})

# ✳️ Ground truth: لكل سؤال ما هي المستندات الـ relevant (بالـ doc_id)
dataset = [
    {
        "question": "What are common retrieval metrics in RAG?",
        "relevant_doc_ids": ["rag_intro"],   # عدّل حسب ملفاتك
    },
    {
        "question": "How do I persist Chroma locally across sessions?",
        "relevant_doc_ids": ["chroma_persist"],
    },
]

def get_doc_ids_from_retrieval(question: str, k: int = 5) -> List[str]:
    docs = retriever.get_relevant_documents(question)
    return [d.metadata.get("doc_id", "") for d in docs][:k]

def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    topk = retrieved[:k]
    rel = sum(1 for doc in topk if doc in relevant)
    return rel / max(1, len(topk))

def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    topk = retrieved[:k]
    rel = sum(1 for doc in topk if doc in relevant)
    return rel / max(1, len(relevant))

def mrr(retrieved: List[str], relevant: List[str]) -> float:
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    # binary gains: 1 لو relevant وإلا 0
    gains = [1.0 if doc in relevant else 0.0 for doc in retrieved[:k]]
    dcg = sum(g / log2(i + 2) for i, g in enumerate(gains))
    # ideal DCG = كل الـ relevant في الأعلى
    ideal_gains = sorted(gains, reverse=True)
    idcg = sum(g / log2(i + 2) for i, g in enumerate(ideal_gains))
    return (dcg / idcg) if idcg > 0 else 0.0

K = 3
p_scores, r_scores, mrr_scores, ndcg_scores = [], [], [], []

for ex in dataset:
    q = ex["question"]
    gold = ex["relevant_doc_ids"]
    ret = get_doc_ids_from_retrieval(q, k=K)

    p = precision_at_k(ret, gold, K)
    r = recall_at_k(ret, gold, K)
    m = mrr(ret, gold)
    n = ndcg_at_k(ret, gold, K)

    p_scores.append(p); r_scores.append(r); mrr_scores.append(m); ndcg_scores.append(n)
    print(f"\nQ: {q}")
    print(f"Retrieved (top-{K}): {ret}")
    print(f"Gold: {gold}")
    print(f"Precision@{K}={p:.2f}  Recall@{K}={r:.2f}  MRR={m:.2f}  nDCG@{K}={n:.2f}")

print("\n==== Averages ====")
print(f"Precision@{K}={sum(p_scores)/len(p_scores):.2f}")
print(f"Recall@{K}   ={sum(r_scores)/len(r_scores):.2f}")
print(f"MRR          ={sum(mrr_scores)/len(mrr_scores):.2f}")
print(f"nDCG@{K}     ={sum(ndcg_scores)/len(ndcg_scores):.2f}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



Q: What are common retrieval metrics in RAG?
Retrieved (top-3): ['rag_intro', 'rag_intro', 'rag_intro']
Gold: ['rag_intro']
Precision@3=1.00  Recall@3=3.00  MRR=1.00  nDCG@3=1.00

Q: How do I persist Chroma locally across sessions?
Retrieved (top-3): ['chroma_persist', 'chroma_persist', 'chroma_persist']
Gold: ['chroma_persist']
Precision@3=1.00  Recall@3=3.00  MRR=1.00  nDCG@3=1.00

==== Averages ====
Precision@3=1.00
Recall@3   =3.00
MRR          =1.00
nDCG@3     =1.00


In [25]:
# ✅ task4_deepeval_final_fixed.py
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# 1️⃣ إعداد قاعدة البيانات (Chroma)
DB_DIR = "chroma_store"
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = Chroma(persist_directory=DB_DIR, embedding_function=emb)
retriever = vs.as_retriever(search_kwargs={"k": 3})

# 2️⃣ إعداد الموديل (OpenAI GPT)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3️⃣ دالة توليد إجابات RAG
def rag_answer(question: str):
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""Use ONLY the context to answer. 
If missing, say "I don't know".
Question: {question}
Context:
{context}
Answer in 3-5 lines:"""
    answer = llm.invoke(prompt).content
    return answer, [d.page_content for d in docs]

# 4️⃣ إعداد الأسئلة للاختبار
questions = [
    "What are common retrieval metrics in RAG?",
    "How can I persist Chroma locally?"
]

cases = []
for q in questions:
    answer, contexts = rag_answer(q)
    case = LLMTestCase(
        input=q,
        actual_output=answer,
        retrieval_context=contexts,  # ✅ التعديل هنا
        expected_output=""
    )
    cases.append(case)

# 5️⃣ المقاييس المستخدمة
metrics = [
    FaithfulnessMetric(),
    AnswerRelevancyMetric()
]

# 6️⃣ التقييم
report = evaluate(test_cases=cases, metrics=metrics)
print(report)


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because there are no contradictions—great job staying true to the retrieval context!, error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because the answer was fully relevant and addressed the question directly without any irrelevant information. Great job staying focused and concise!, error: None)

For test case:

  - input: What are common retrieval metrics in RAG?
  - actual output: Common retrieval metrics in RAG include Precision@K, Recall@K, Mean Reciprocal Rank (MRR), and normalized Discounted Cumulative Gain (nDCG). These metrics are used to evaluate the effectiveness of the retrieval component in the retrieval-augmented generation process.
  - expected output: 
  - context: None
  - retrieval context: ['RAG combines retrieval with generation to ground LLM answers.

⚠ WARNING: No hyperparameters logged.
» ]8;id=535603;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.57s | token cost: 0.016654000000000002 USD)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions—great job staying true to the retrieval context!', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.004702, verbose_logs='Truths (limit=None):\n[\n    "RAG combines retrieval with generation to ground LLM answers.",\n    "Common retrieval metrics include Precision@K, Recall@K, MRR, and nDCG."\n] \n \nClaims:\n[\n    "Common retrieval metrics in RAG include Precision@K, Recall@K, Mean Reciprocal Rank (MRR), and normalized Discounted Cumulative Gain (nDCG).",\n    "These metrics are used to evaluate the effectiveness of the retrieval component in the retrieval-augmented generation process."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(

In [26]:
# ✅ rag_test_chat.py
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

# إعداد الـ LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# تحميل قاعدة البيانات
DB_DIR = "chroma_store"
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = Chroma(persist_directory=DB_DIR, embedding_function=emb)
retriever = vs.as_retriever(search_kwargs={"k": 3})

# تنسيق البرومبت
prompt = ChatPromptTemplate.from_template("""
Use ONLY the following context to answer the question clearly:
{context}
Question: {question}
Answer concisely.
""")

# تحويل المستندات لنص واحد
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# بناء سلسلة RAG (Retriever + Prompt + Model)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# تجربة تفاعلية
print("🚀 RAG Chat Ready! Type 'exit' to stop.\n")
while True:
    q = input("You: ")
    if q.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break
    ans = rag_chain.invoke(q)
    print("AI:", ans.content, "\n")


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🚀 RAG Chat Ready! Type 'exit' to stop.

AI: Artificial intelligence (AI) is the simulation of human intelligence processes by machines. 

AI: To learn AI, one can study the principles of artificial intelligence, including machine learning, data analysis, and algorithms. Engaging in practical projects, utilizing online courses, and participating in AI communities can also enhance understanding and skills in AI. 

AI: RAG (Retrieval Augmented Generation) is a method that combines retrieval and generation, retrieving relevant data from a database to enhance the accuracy of language models. 

AI: RAG integrates retrieval and generation to enhance LLM responses, utilizing metrics like Precision@K, Recall@K, MRR, and nDCG for evaluation. 

AI: Goodbye! 

AI: Goodbye! 

👋 Goodbye!
